## Continued Pretraining Phase

In [1]:
from src.core.settings import settings
CPT_DATASET_SEED = 42
INS_FT_DATASET_SEED = 123

In [2]:
from src.get_training_data.pretraining_data import download_multi_datasets_async, split_and_save_cpt_dataset

d:\finetuning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\finetuning\.venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
cpt_dataset = await download_multi_datasets_async(seed = CPT_DATASET_SEED)

print(cpt_dataset)

cpt_dataset = await split_and_save_cpt_dataset(cpt_dataset)

print(cpt_dataset)

Loading Dataset Infos from C:\Users\blaks\.cache\huggingface\modules\datasets_modules\datasets\joelniklaus--Multi_Legal_Pile\453fcdf95171db34c9daf28f359d524754b752b9f6b8ee6f3e66b0865ebc5837
Loading Dataset Infos from C:\Users\blaks\.cache\huggingface\modules\datasets_modules\datasets\joelniklaus--Multi_Legal_Pile\453fcdf95171db34c9daf28f359d524754b752b9f6b8ee6f3e66b0865ebc5837


DatasetDict({
    contracts: Dataset({
        features: ['text'],
        num_rows: 20
    })
    caselaw: Dataset({
        features: ['text'],
        num_rows: 20
    })
    general: Dataset({
        features: ['text'],
        num_rows: 5
    })
})
Saving pretraining dataset to %s training_data\cpt_data


Saving the dataset (1/1 shards): 100%|██████████| 7/7 [00:00<00:00, 215.47 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 38
    })
    val: Dataset({
        features: ['text'],
        num_rows: 7
    })
})


In [ ]:
from src.core.training.common import tokenize_dataset, load_training_config
from src.core.training.cpt import get_model_and_tokenizer_async, chunk_tokens_cpt

In [ ]:
model, tokenizer = await get_model_and_tokenizer_async()

In [ ]:
cpt_train_data = cpt_dataset['train']
cpt_val_data = cpt_dataset['val']

In [ ]:
cpt_train_data_tokenized = tokenize_dataset(cpt_train_data, tokenizer, chunk_tokens_cpt)
cpt_val_data_tokenized = tokenize_dataset(cpt_val_data, tokenizer, chunk_tokens_cpt)

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArguments

cpt_training_config = load_training_config("TrainingArgsCPT")

cpt_trainer = UnslothTrainer(
    model = model,
    train_dataset = cpt_train_data_tokenized,
    eval_dataset=cpt_val_data_tokenized,
    tokenizer = tokenizer,
    args = UnslothTrainingArguments(
        **cpt_training_config
    ),
)

cpt_trainer_results = cpt_trainer.train()

In [ ]:
####################
trainer_eval_results = cpt_trainer_results.evaluate()
####################

In [ ]:
save_model_path = settings.CPT_MODEL_PATH
save_model_path.mkdir(parents = True, exist_ok = True)
model.save_pretrained(save_model_path)
tokenizer.save_pretrained(save_model_path)

## Instruction Finetuning

In [ ]:
from src.get_training_data.instruction_finetuning_data import download_contracts_dataset, submit_summarization_batch_job, submit_qna_batch_job, parse_results

In [ ]:
dataset = await download_contracts_dataset(seed = INS_FT_DATASET_SEED, split = True)
dataset

In [ ]:
summarization_job = await submit_summarization_batch_job(dataset = dataset)
summarization_job_name = summarization_job.name

In [ ]:
summary_results, summary_errorneous_keys = await parse_results(
    job_name = summarization_job_name,
    text_column_name= "summary",
    text_column_token_count_name = "summary_llama_tokens",
    batch_results_path = settings.INS_FT_MLP_DATA_DIR_SUMM / settings.INS_FT_MLP_BATCH_RESULTS,
    key_column = "key",
    dataset_limit = None
)

In [ ]:
from src.core.training.ift import load_model_and_tokenizer_async, chunk_tokens_ift, process_summary_dataset, process_entity_dataset

model, tokenizer = await load_model_and_tokenizer_async()

In [ ]:
summary_dataset = summary_results.map(process_summary_dataset)

In [ ]:
summary_dataset = summary_dataset.map(
    chunk_tokens_ift,
    batched = True,
    batch_size = 8,
    fn_kwargs = {"tokenizer": tokenizer, "task": "summary"},
    remove_columns = summary_dataset.column_names
)

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only

ift_trainer_args = load_training_config("TrainerIFT")
ift_training_args = load_training_config("TrainingArgsIFT")

ift_summarization_trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = summary_dataset,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    **ift_trainer_args,
    args = SFTConfig(
        **ift_training_args
    ),
)

ift_summarization_trainer = train_on_responses_only(
    ift_summarization_trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

ift_summarization_trainer_stats = ift_summarization_trainer.train()

In [ ]:
clause_detection_job = await submit_qna_batch_job(dataset = dataset)
clause_detection_job_name = clause_detection_job.name

In [ ]:
clause_detection_job_results, clause_detection_errorneous_keys = await parse_results(
    job_name = clause_detection_job_name,
    text_column_name= "entities",
    text_column_token_count_name = "entities_llama_tokens",
    batch_results_path = settings.INS_FT_MLP_DATA_DIR_QNA / settings.INS_FT_MLP_BATCH_RESULTS,
    key_column = "key",
    dataset_limit = None
)

In [ ]:
clause_detection_dataset, erroneous_entities = process_entity_dataset(clause_detection_job_results)

In [ ]:
clause_detection_errorneous_keys.update(erroneous_entities)

In [ ]:
##########################  # one processing function for selecting questions/answers

clause_detection_dataset = clause_detection_dataset

##########################

In [ ]:
clause_detection_dataset = clause_detection_dataset.map(
    chunk_tokens_ift,
    batched = True,
    batch_size = 8,
    fn_kwargs = {"tokenizer": tokenizer, "task": "qa"},
    remove_columns = clause_detection_dataset.column_names
)

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only

ift_trainer_args = load_training_config("TrainerIFT")
ift_training_args = load_training_config("TrainingArgsIFT")

ift_clause_detection_trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = clause_detection_dataset,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    **ift_trainer_args,
    args = SFTConfig(
        **ift_training_args
    ),
)

ift_clause_detection_trainer = train_on_responses_only(
    ift_clause_detection_trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

ift_clause_detection_trainer_stats = ift_clause_detection_trainer.train()

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatoraForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized_instruct_dataset_train,
    # dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

trainer_stats = trainer.train()